In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# CONFIG
# ============================================================

CSV_PATH = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\Analysis_v2\analysis_dataset.csv"

OUTPUT_DIR = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\Analysis_v2\EDA"

TAB_DIR = os.path.join(OUTPUT_DIR, "tables")
FIG_DIR = os.path.join(OUTPUT_DIR, "figures")

FIG_NUMERIC_DIR = os.path.join(FIG_DIR, "01_numeric_variables")
FIG_CATEGORICAL_DIR = os.path.join(FIG_DIR, "02_categorical_variables")
FIG_OUTCOME_CONFOUNDER_DIR = os.path.join(FIG_DIR, "03_outcome_vs_confounder")
FIG_CORR_DIR = os.path.join(FIG_DIR, "04_correlations")

for d in [
    TAB_DIR,
    FIG_NUMERIC_DIR,
    FIG_CATEGORICAL_DIR,
    FIG_OUTCOME_CONFOUNDER_DIR,
    FIG_CORR_DIR
]:
    os.makedirs(d, exist_ok=True)

# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv(CSV_PATH)
print(df)

df_panel = df.copy()

df_nhda = (
    df.sort_values(["nhda_id", "years_since_construction_start"])
    .drop_duplicates(subset="nhda_id")
    .copy()
)

print("\nPanel-level shape:", df_panel.shape)
print("NHDA-level shape:", df_nhda.shape)

# ============================================================
# VARIABLE GROUPS
# ============================================================

outcome_cols = [
    "difference_LST",
    "difference_NDVI"
]

time_cols = [
    "years_since_construction_start"
]

morphology_cols = [
    # subclass share differences
    'diff_res_subclass_share_mfh_ab',
    'diff_res_subclass_share_sbd',
    'diff_res_subclass_share_tb',
    'diff_res_subclass_share_sfh_db',

    # morphology predictors
    'reldiff_building_volume_density',
    'reldiff_building_density',
    'reldiff_avg_building_footprint',
    'reldiff_avg_building_height',
    'reldiff_built_up_ratio',

]

# baseline_cols = [
#     "pre_construction_difference_LST",
#     "pre_construction_difference_NDVI"
# ]

categorical_cols = [
    "nhda_degurba_code",
    "nhda_sur_class_2021_short",
    "years_since_construction_start"
]

outcome_cols = [c for c in outcome_cols if c in df_panel.columns]
time_cols = [c for c in time_cols if c in df_panel.columns]
morphology_cols = [c for c in morphology_cols if c in df_nhda.columns]
# baseline_cols = [c for c in baseline_cols if c in df_nhda.columns]
categorical_cols = [c for c in categorical_cols if c in df_nhda.columns]

panel_numeric_cols = outcome_cols + time_cols
nhda_numeric_cols = morphology_cols 

# ============================================================
# BASIC OVERVIEW
# ============================================================

overview_panel = pd.DataFrame({
    "column": df_panel.columns,
    "dtype": df_panel.dtypes.astype(str).values,
    "n_missing": df_panel.isna().sum().values,
    "pct_missing": (df_panel.isna().mean() * 100).round(2).values,
    "n_unique": df_panel.nunique(dropna=True).values
})

overview_nhda = pd.DataFrame({
    "column": df_nhda.columns,
    "dtype": df_nhda.dtypes.astype(str).values,
    "n_missing": df_nhda.isna().sum().values,
    "pct_missing": (df_nhda.isna().mean() * 100).round(2).values,
    "n_unique": df_nhda.nunique(dropna=True).values
})

overview_panel.to_csv(os.path.join(TAB_DIR, "overview_panel.csv"), index=False)
overview_nhda.to_csv(os.path.join(TAB_DIR, "overview_nhda.csv"), index=False)

# ============================================================
# 1. NUMERIC VARIABLES
# ============================================================

print("\nBLOCK 1: Numeric variables")

desc_panel = df_panel[panel_numeric_cols].describe().T
desc_panel["missing"] = df_panel[panel_numeric_cols].isna().sum()
desc_panel["pct_missing"] = (df_panel[panel_numeric_cols].isna().mean() * 100).round(2)
desc_panel.to_csv(os.path.join(TAB_DIR, "01_descriptive_statistics_panel_numeric.csv"))

desc_nhda = df_nhda[nhda_numeric_cols].describe().T
desc_nhda["missing"] = df_nhda[nhda_numeric_cols].isna().sum()
desc_nhda["pct_missing"] = (df_nhda[nhda_numeric_cols].isna().mean() * 100).round(2)
desc_nhda.to_csv(os.path.join(TAB_DIR, "01_descriptive_statistics_nhda_numeric.csv"))

for col in panel_numeric_cols:
    # plt.figure(figsize=(7, 5))
    # plt.hist(df_panel[col].dropna(), bins=40)
    # plt.title(f"Panel-level distribution of {col}")
    # plt.xlabel(col)
    # plt.ylabel("Frequency")
    plt.figure(figsize=(7, 5))

    # Farben und Titel für Outcomes
    if col == "difference_LST":
        color = "#a4133c"
        title = "Distribution of ΔLST"
        xlabel = "ΔLST"
    elif col == "difference_NDVI":
        color = "#2d6a4f"
        title = "Distribution of ΔNDVI"
        xlabel = "ΔNDVI"
    else:
        color = "steelblue"
        title = f"Panel-level distribution of {col}"
        xlabel = col

    plt.hist(
        df_panel[col].dropna(),
        bins=40,
        color=color,
        edgecolor="white",
        linewidth=0.6,
        alpha=0.9
    )

    plt.title(title, fontsize=14, fontweight="bold")
    plt.xlabel(xlabel, fontsize=12)
    plt.ylabel("Frequency", fontsize=12)

    plt.grid(axis="y", alpha=0.25)

    # schöneres Layout
    ax = plt.gca()
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    plt.savefig(os.path.join(FIG_NUMERIC_DIR, f"hist_panel_{col}.png"), dpi=300)
    plt.close()

    plt.figure(figsize=(7, 4))
    plt.boxplot(df_panel[col].dropna(), vert=False)
    plt.title(f"Panel-level boxplot of {col}")
    plt.xlabel(col)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_NUMERIC_DIR, f"boxplot_panel_{col}.png"), dpi=300)
    plt.close()

# for col in nhda_numeric_cols:
#     plt.figure(figsize=(7, 5))
#     plt.hist(df_nhda[col].dropna(), bins=40)
#     plt.title(f"NHDA-level distribution of {col}")
#     plt.xlabel(col)
#     plt.ylabel("Number of NHDAs")
#     plt.tight_layout()
#     plt.savefig(os.path.join(FIG_NUMERIC_DIR, f"hist_nhda_{col}.png"), dpi=300)
#     plt.close()

#     plt.figure(figsize=(7, 4))
#     plt.boxplot(df_nhda[col].dropna(), vert=False)
#     plt.title(f"NHDA-level boxplot of {col}")
#     plt.xlabel(col)
#     plt.tight_layout()
#     plt.savefig(os.path.join(FIG_NUMERIC_DIR, f"boxplot_nhda_{col}.png"), dpi=300)
#     plt.close()
for col in panel_numeric_cols:
    if col == "difference_LST":
        color = "#a4133c"
        title = "Distribution of ΔLST"
        xlabel = "ΔLST"
    elif col == "difference_NDVI":
        color = "#2d6a4f"
        title = "Distribution of ΔNDVI"
        xlabel = "ΔNDVI"
    else:
        color = "steelblue"
        title = f"Panel-level distribution of {col}"
        xlabel = col

    fig, ax = plt.subplots(figsize=(7, 3.5))

    ax.hist(
        df_panel[col].dropna().to_numpy(),
        bins=40,
        color=color
    )

    ax.set_title(title, fontsize=16, fontweight="bold")
    ax.set_xlabel(xlabel, fontsize=14)
    ax.set_ylabel("Frequency", fontsize=14)

    fig.tight_layout()
    fig.savefig(os.path.join(FIG_NUMERIC_DIR, f"hist_panel_{col}.png"), dpi=200)
    plt.close(fig)

# Sample size by year
sample_by_year = (
    df_panel.groupby("years_since_construction_start")
    .agg(
        n_rows=("nhda_id", "size"),
        n_nhda=("nhda_id", "nunique"),
        n_LST=("difference_LST", "count"),
        n_NDVI=("difference_NDVI", "count")
    )
    .reset_index()
)

sample_by_year.to_csv(os.path.join(TAB_DIR, "01_sample_size_by_year.csv"), index=False)

# Outcome over time
for outcome in outcome_cols:
    summary_time = (
        df_panel.groupby("years_since_construction_start")[outcome]
        .agg(["mean", "median", "std", "count"])
        .reset_index()
    )

    summary_time.to_csv(
        os.path.join(TAB_DIR, f"01_{outcome}_by_year.csv"),
        index=False
    )

    plt.figure(figsize=(7, 5))
    plt.plot(
        summary_time["years_since_construction_start"],
        summary_time["mean"],
        marker="o"
    )
    plt.title(f"Mean {outcome} by years since construction start")
    plt.xlabel("Years since construction start")
    plt.ylabel(outcome)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_NUMERIC_DIR, f"mean_{outcome}_by_year.png"), dpi=300)
    plt.close()

# ============================================================
# 2. CATEGORICAL VARIABLES
# ============================================================

print("BLOCK 2: Categorical variables")

for col in categorical_cols:
    counts = (
        df_nhda[col]
        .value_counts(dropna=False)
        .reset_index()
    )

    counts.columns = [col, "count"]
    counts["percent"] = (100 * counts["count"] / counts["count"].sum()).round(2)

    counts.to_csv(
        os.path.join(TAB_DIR, f"02_counts_nhda_{col}.csv"),
        index=False
    )

    plt.figure(figsize=(8, 5))
    plt.bar(counts[col].astype(str), counts["count"])
    plt.title(f"NHDA-level distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Number of NHDAs")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_CATEGORICAL_DIR, f"barplot_nhda_{col}.png"), dpi=300)
    plt.close()

# Cross-tab between DEGURBA and surrounding land cover
if "nhda_degurba_code" in df_nhda.columns and "nhda_sur_class_2021_short" in df_nhda.columns:
    ctab = pd.crosstab(
        df_nhda["nhda_degurba_code"],
        df_nhda["nhda_sur_class_2021_short"],
        margins=True
    )

    ctab.to_csv(os.path.join(TAB_DIR, "02_crosstab_degurba_surrounding_landcover.csv"))

# ============================================================
# 3. OUTCOME VS CONFOUNDER
# ============================================================
outcome_labels = {
    "difference_LST": "ΔLST (°C)",
    "difference_NDVI": "ΔNDVI"
}

cat_labels = {
    "nhda_sur_class_2021_short": "Surrounding Landcover (CLC-L1)",
    "nhda_degurba_code": "Degree of Urbanisation",
    "years_since_construction_start": "Years since construction start"
}

landcover_labels = {
    1: "Artificial Surface",
    2: "Agricultural Areas",
    3: "Forests and semi-natural areas"
}

landcover_colors = {
    1: "#e6004d",
    2: "#ffffa8",
    3: "#80a000"
}

degurba_colors = {
    130: {"class": "Urban Centre",                 "color": "#FF0000"},
    223: {"class": "Dense Urban Cluster",          "color": "#993300"},
    222: {"class": "Semi-dense Urban Cluster",     "color": "#CC9900"},
    221: {"class": "Suburban Grid Cell",           "color": "#FFFF00"},
    313: {"class": "Rural Cluster",                "color": "#336633"},
    312: {"class": "Low Density Rural Grid Cell",  "color": "#99CC66"},
    311: {"class": "Very Low Density Grid Cell",   "color": "#CCFF99"}
}

# Boxplots: outcomes by categorical confounders
for outcome in outcome_cols:
    for cat in categorical_cols:
        if outcome in df_panel.columns and cat in df_panel.columns:

            plot_df = df_panel[[outcome, cat]].dropna().copy()

            # Labels und Farben je Confounder
            if cat == "nhda_sur_class_2021_short":
                order = [1, 2, 3]
                labels = [landcover_labels[x] for x in order]
                colors = [landcover_colors[x] for x in order]

            elif cat == "nhda_degurba_code":
                order = [130, 223, 222, 221, 313, 312, 311, 310]
                order = [x for x in order if x in plot_df[cat].unique()]
                labels = [str(x) for x in order]
                colors = [degurba_colors[x]["color"] for x in order]

            else:
                order = sorted(plot_df[cat].unique())
                labels = [str(x) for x in order]
                colors = ["lightgrey"] * len(order)

            data = [
                plot_df.loc[plot_df[cat] == x, outcome].dropna().to_numpy()
                for x in order
            ]

            fig, ax = plt.subplots(figsize=(10, 8))

            bp = ax.boxplot(
                data,
                patch_artist=True,
                tick_labels=labels,
                showfliers=False
            )

            # Einzelwerte / Streuung als jittered points anzeigen
            for i, values in enumerate(data, start=1):
                x_jitter = np.random.normal(loc=i, scale=0.05, size=len(values))

                ax.scatter(
                    x_jitter,
                    values,
                    color="black",
                    alpha=0.15,
                    s=5, 
                    linewidths=0
                )
            for patch, color in zip(bp["boxes"], colors):
                patch.set_facecolor(color)
                patch.set_alpha(0.85)
                patch.set_edgecolor("black")

            for median in bp["medians"]:
                median.set_color("black")
                median.set_linewidth(1.5)

            ax.set_title(
                f"{outcome_labels.get(outcome, outcome)} by {cat_labels.get(cat, cat)}",
                fontsize=20,
                fontweight="bold"
            )

            ax.set_xlabel(cat_labels.get(cat, cat), fontsize=15)
            ax.set_ylabel(outcome_labels.get(outcome, outcome), fontsize=15)

            # ax.tick_params(axis="x", labelrotation=45, labelsize=10)
            if cat == "nhda_sur_class_2021_short":
                ax.tick_params(axis="x", labelrotation=0, labelsize=14)
            else:
                ax.tick_params(axis="x", labelrotation=0, labelsize=14)
            ax.tick_params(axis="y", labelsize=14)

            ax.grid(axis="y", alpha=0.25)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

            fig.tight_layout()
            fig.savefig(
                os.path.join(
                    FIG_OUTCOME_CONFOUNDER_DIR,
                    f"boxplot_{outcome}_by_{cat}.png"
                ),
                dpi=300
            )
            plt.close(fig)

print("BLOCK 3: Outcome vs confounder")


# Scatterplots: outcomes vs numerical NHDA-level predictors/confounders
for outcome in outcome_cols:
    for x in nhda_numeric_cols:
        if outcome in df_panel.columns and x in df_panel.columns:
            plot_df = df_panel[[x, outcome]].dropna()

            plt.figure(figsize=(7, 5))
            plt.scatter(plot_df[x], plot_df[outcome], alpha=0.18, s=10)
            plt.title(f"{outcome} vs NHDA-level {x}")
            plt.xlabel(x)
            plt.ylabel(outcome)
            plt.tight_layout()
            plt.savefig(
                os.path.join(
                    FIG_OUTCOME_CONFOUNDER_DIR,
                    f"scatter_{outcome}_vs_{x}.png"
                ),
                dpi=300
            )
            plt.close()

# Panel-level outcome relationships
if "difference_LST" in df_panel.columns and "difference_NDVI" in df_panel.columns:
    plot_df = df_panel[["difference_NDVI", "difference_LST"]].dropna()

    plt.figure(figsize=(7, 5))
    plt.scatter(plot_df["difference_NDVI"], plot_df["difference_LST"], alpha=0.18, s=10)
    plt.title("difference_LST vs difference_NDVI")
    plt.xlabel("difference_NDVI")
    plt.ylabel("difference_LST")
    plt.tight_layout()
    plt.savefig(
        os.path.join(FIG_OUTCOME_CONFOUNDER_DIR, "scatter_difference_LST_vs_difference_NDVI.png"),
        dpi=300
    )
    plt.close()

# ============================================================
# 4. CORRELATIONS + MULTICOLLINEARITY
# ============================================================

print("BLOCK 4: Correlations and multicollinearity")

pretty_names = {
    "reldiff_built_up_ratio": "Rel. Δ Built-up Ratio [%]",
    "reldiff_building_volume_density": "Rel. Δ Building Volume Density [m³/m²]",
    "reldiff_building_density": "Rel. Δ Building Density [1/ha]",
    "reldiff_avg_building_height": "Rel. Δ Avg. Building Height [m]",
    "reldiff_avg_building_footprint": "Rel. Δ Avg. Building Footprint [m²]",

    "SFH-DB_share_difference": "Δ Share SFH-DB [%]",
    "SBD_share_difference": "Δ Share SBD [%]",
    "TB_share_difference": "Δ Share TB [%]",
    "MFH_AB_share_difference": "Δ Share MFH-AB [%]",

    "pre_construction_difference_LST": "Pre-construction ΔLST",
    "pre_construction_difference_NDVI": "Pre-construction ΔNDVI"
}

# variables for correlation matrix
corr_cols = morphology_cols 
corr_cols = [c for c in corr_cols if c in df_nhda.columns]

# Pearson correlation matrix
corr_nhda = df_nhda[corr_cols].corr()

corr_nhda.to_csv(
    os.path.join(TAB_DIR, "04_correlation_matrix_nhda_numeric.csv")
)

# Plot
fig, ax = plt.subplots(figsize=(12, 10))

im = ax.imshow(
    corr_nhda,
    cmap="RdBu_r",
    vmin=-1,
    vmax=1
)

# Correlation coefficients in cells
for i in range(len(corr_nhda)):
    for j in range(len(corr_nhda)):

        value = corr_nhda.iloc[i, j]

        text_color = "white" if abs(value) >= 0.5 else "black"

        ax.text(
            j,
            i,
            f"{value:.2f}",
            ha="center",
            va="center",
            fontsize=12,
            color=text_color
        )

# Pretty labels
labels = [
    pretty_names.get(col, col)
    for col in corr_nhda.columns
]

ax.set_xticks(np.arange(len(labels)))
ax.set_yticks(np.arange(len(labels)))

ax.set_xticklabels(
    labels,
    rotation=25,
    ha="right",
    rotation_mode="anchor",
    fontsize=10
)

ax.set_yticklabels(
    labels,
    fontsize=10
)

# Colorbar
# cbar = plt.colorbar(im, ax=ax)
# cbar.set_label("Pearson correlation", fontsize=12)
cbar = plt.colorbar(
    im,
    ax=ax,
    shrink=0.6,      # Höhe der Colorbar
    aspect=20,       # Breite-Höhe-Verhältnis
    pad=0.02         # Abstand zur Matrix
)

cbar.set_label(
    "Pearson correlation",
    fontsize=11
)

cbar.ax.tick_params(labelsize=9)
# Title
# ax.set_title(
#     "Correlation matrix of morphological and baseline variables",
#     fontsize=16,
#     fontweight="bold"
# )

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIG_CORR_DIR,
        "correlation_matrix_nhda_numeric_v2.jpg"
    ),
    dpi=300
)

plt.close()


# Simple VIF calculation without statsmodels
vif_rows = []

vif_data = df_nhda[corr_cols].dropna().copy()

for target in corr_cols:
    predictors = [c for c in corr_cols if c != target]

    if len(predictors) < 1:
        continue

    X = vif_data[predictors].values
    y = vif_data[target].values

    X = np.column_stack([np.ones(len(X)), X])

    try:
        beta, *_ = np.linalg.lstsq(X, y, rcond=None)
        y_hat = X @ beta

        ss_res = np.sum((y - y_hat) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)

        r2 = 1 - ss_res / ss_tot if ss_tot != 0 else np.nan
        vif = 1 / (1 - r2) if r2 < 1 else np.inf

        vif_rows.append({
            "variable": target,
            "r_squared_against_other_predictors": r2,
            "VIF": vif
        })

    except Exception:
        vif_rows.append({
            "variable": target,
            "r_squared_against_other_predictors": np.nan,
            "VIF": np.nan
        })

vif_df = pd.DataFrame(vif_rows).sort_values("VIF", ascending=False)
vif_df.to_csv(os.path.join(TAB_DIR, "04_vif_nhda_numeric_predictors.csv"), index=False)

# Zero summary for share differences
share_diff_cols = [
    "MFH-AB_share_difference",
    "SBD_share_difference",
    "TB_share_difference",
    "SFH-DB_share_difference"
]

zero_summary = []

for col in share_diff_cols:
    if col in df_nhda.columns:
        n_total = len(df_nhda)
        n_zero = (df_nhda[col] == 0).sum()
        n_nonzero = (df_nhda[col] != 0).sum()

        zero_summary.append({
            "variable": col,
            "n_total_nhda": n_total,
            "n_zero": n_zero,
            "pct_zero": round(100 * n_zero / n_total, 2),
            "n_nonzero": n_nonzero,
            "pct_nonzero": round(100 * n_nonzero / n_total, 2)
        })

zero_summary = pd.DataFrame(zero_summary)
zero_summary.to_csv(
    os.path.join(TAB_DIR, "04_share_difference_zero_summary_nhda.csv"),
    index=False
)

# ============================================================
# CATEGORICAL ASSOCIATION: KRUSKAL-WALLIS + EPSILON SQUARED
# ============================================================

from scipy.stats import kruskal

print("Categorical associations: Kruskal-Wallis + epsilon squared")

cat_assoc_cols = [
    "nhda_degurba_code",
    "nhda_sur_class_2021_short"
]

cat_assoc_cols = [c for c in cat_assoc_cols if c in df_nhda.columns]

numeric_assoc_cols = corr_cols.copy()
numeric_assoc_cols = [
    c for c in numeric_assoc_cols
    if c != "years_since_construction_start"
]

cat_assoc_labels = {
    "nhda_degurba_code": "DEGURBA",
    "nhda_sur_class_2021_short": "CLC-L1"
}

pretty_names = {
    "reldiff_built_up_ratio": "Rel. Δ Built-up Ratio",
    "reldiff_building_volume_density": "Rel. Δ Building Volume Density",
    "reldiff_building_density": "Rel. Δ Building Density",
    "reldiff_avg_building_height": "Rel. Δ Avg. Building Height",
    "reldiff_avg_building_footprint": "Rel. Δ Avg. Building Footprint",
    "MFH_AB_share_difference": "Δ Share MFH-AB",
    "SBD_share_difference": "Δ Share SBD",
    "SFH-DB_share_difference": "Δ Share SFH-DB",
    "TB_share_difference": "Δ Share TB",
    "pre_construction_difference_LST": "Pre-construction ΔLST",
    "pre_construction_difference_NDVI": "Pre-construction ΔNDVI"
}

assoc_rows = []

for cat in cat_assoc_cols:
    for var in numeric_assoc_cols:

        tmp = df_nhda[[cat, var]].dropna().copy()

        groups = [
            group[var].values
            for _, group in tmp.groupby(cat)
            if len(group[var].dropna()) > 1
        ]

        if len(groups) < 2:
            H = np.nan
            p = np.nan
            epsilon_sq = np.nan
        else:
            H, p = kruskal(*groups)

            n = len(tmp)
            k = len(groups)

            # Epsilon squared for Kruskal-Wallis
            epsilon_sq = (H - k + 1) / (n - k)

            # avoid small negative values due to sampling/statistical noise
            epsilon_sq = max(0, epsilon_sq)

        assoc_rows.append({
            "categorical_variable": cat,
            "categorical_label": cat_assoc_labels.get(cat, cat),
            "numeric_variable": var,
            "numeric_label": pretty_names.get(var, var),
            "kruskal_H": H,
            "p_value": p,
            "epsilon_squared": epsilon_sq
        })

assoc_df = pd.DataFrame(assoc_rows)

assoc_df.to_csv(
    os.path.join(TAB_DIR, "04_kruskal_epsilon_squared_categorical_associations.csv"),
    index=False
)

# Matrix for heatmap
epsilon_matrix = assoc_df.pivot(
    index="categorical_label",
    columns="numeric_label",
    values="epsilon_squared"
)

fig, ax = plt.subplots(figsize=(13, 4.5))

im = ax.imshow(
    epsilon_matrix,
    cmap="Blues",
    vmin=0,
    vmax=max(0.14, np.nanmax(epsilon_matrix.values))
)

# values in cells
for i in range(epsilon_matrix.shape[0]):
    for j in range(epsilon_matrix.shape[1]):
        value = epsilon_matrix.iloc[i, j]
        if not np.isnan(value):
            ax.text(
                j,
                i,
                f"{value:.2f}",
                ha="center",
                va="center",
                fontsize=10,
                color="black"
            )

ax.set_xticks(range(epsilon_matrix.shape[1]))
ax.set_yticks(range(epsilon_matrix.shape[0]))

ax.set_xticklabels(epsilon_matrix.columns, rotation=45, ha="right")
ax.set_yticklabels(epsilon_matrix.index)

# ax.set_title(
#     "Association between categorical context variables and numeric predictors",
#     fontsize=15,
#     fontweight="bold"
# )

cbar = plt.colorbar(
    im,
    ax=ax,
    shrink=0.5,
    aspect=20,
    pad=0.02
)

cbar.set_label(
    r"$\varepsilon^2$",
    fontsize=11
)

cbar.ax.tick_params(labelsize=9)

fig.tight_layout()

fig.savefig(
    os.path.join(FIG_CORR_DIR, "kruskal_epsilon_squared_categorical_associations.jpg"),
    dpi=300
)

plt.close(fig)

# ============================================================
# COMPLETION
# ============================================================

print("\nEDA complete.")
print(f"Tables saved to: {TAB_DIR}")
print(f"Figures saved to: {FIG_DIR}")

# Variance Inflation Factor

In [ ]:
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor

# =============================================================================
# VIF FUNCTION
# =============================================================================

def calculate_vif(df, variables):
    """
    Calculate Variance Inflation Factors (VIF) for a list of variables.
    """
    X = df[variables].dropna()

    vif_df = pd.DataFrame({
        "Variable": X.columns,
        "VIF": [
            variance_inflation_factor(X.values, i)
            for i in range(X.shape[1])
        ]
    })

    return vif_df.sort_values("VIF", ascending=False).reset_index(drop=True)


# =============================================================================
# VARIABLES
# =============================================================================

lst_variables = [
    'reldiff_building_volume_density',
    'reldiff_building_density',
    'reldiff_avg_building_footprint',
    'reldiff_avg_building_height',
    'reldiff_built_up_ratio',
    'diff_res_subclass_share_mfh_ab',
    'difference_NDVI'
]

ndvi_variables = [
    'reldiff_building_volume_density',
    'reldiff_building_density',
    'reldiff_avg_building_footprint',
    'reldiff_avg_building_height',
    'reldiff_built_up_ratio',
    'diff_res_subclass_share_mfh_ab'
]

# =============================================================================
# CALCULATE VIF
# =============================================================================

print("=" * 60)
print("LST MODEL")
print("=" * 60)
vif_lst = calculate_vif(df_panel, lst_variables)
print(vif_lst)

print("\n" + "=" * 60)
print("NDVI MODEL")
print("=" * 60)
vif_ndvi = calculate_vif(df_panel, ndvi_variables)
print(vif_ndvi)

In [ ]:
import itertools
import numpy as np
import pandas as pd

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

# =============================================================================
# CONFIGURATION
# =============================================================================

VIF_THRESHOLD = 4.0
MIN_VARIABLES = 2


lst_variables = [
    "reldiff_built_up_ratio",
    "reldiff_building_volume_density",
    "reldiff_building_density",
    "reldiff_avg_building_footprint",
    "reldiff_avg_building_height",
    "diff_res_subclass_share_mfh_ab",
    "difference_NDVI"
]


ndvi_variables = [
    "reldiff_built_up_ratio",
    "reldiff_building_volume_density",
    "reldiff_building_density",
    "reldiff_avg_building_footprint",
    "reldiff_avg_building_height",
    "diff_res_subclass_share_mfh_ab"
]


# =============================================================================
# VIF CALCULATION
# =============================================================================

def calculate_vif(df, variables):
    """
    Calculate VIF values for one predictor combination.

    A constant is included in the auxiliary regressions but is not returned
    in the VIF table.
    """
    X = df[list(variables)].copy()

    # Convert all predictors to numeric
    X = X.apply(pd.to_numeric, errors="coerce")

    # Remove rows containing missing or infinite values
    X = X.replace([np.inf, -np.inf], np.nan).dropna()

    if len(X) == 0:
        return None

    # Variables without variation cannot be used
    if (X.nunique() <= 1).any():
        return None

    X_with_constant = add_constant(X, has_constant="add")

    vif_values = []

    # Start at 1 to exclude the constant
    for i, variable in enumerate(X.columns, start=1):
        try:
            vif = variance_inflation_factor(
                X_with_constant.values,
                i
            )
        except Exception:
            vif = np.inf

        vif_values.append({
            "Variable": variable,
            "VIF": vif
        })

    return pd.DataFrame(vif_values)


# =============================================================================
# TEST ALL VARIABLE COMBINATIONS
# =============================================================================

def test_all_vif_combinations(
    df,
    variables,
    model_name,
    threshold=4.0,
    min_variables=2
):
    """
    Test all possible predictor combinations.

    Ranking:
    1. Largest number of variables
    2. Lowest maximum VIF
    3. Lowest mean VIF
    """
    missing_variables = [
        variable for variable in variables
        if variable not in df.columns
    ]

    if missing_variables:
        raise KeyError(
            f"{model_name}: These variables are missing:\n"
            + "\n".join(missing_variables)
        )

    # Use one common complete-case dataset for all combinations.
    # This ensures that all combinations are compared using the same rows.
    model_df = (
        df[variables]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    print("\n" + "=" * 80)
    print(f"{model_name}")
    print("=" * 80)
    print(f"Complete observations used: {len(model_df)}")
    print(f"Candidate variables: {len(variables)}")

    combination_results = []
    detailed_vif_results = []

    # Test combinations from largest to smallest
    for number_variables in range(
        len(variables),
        min_variables - 1,
        -1
    ):
        for combination in itertools.combinations(
            variables,
            number_variables
        ):
            vif_table = calculate_vif(
                model_df,
                combination
            )

            if vif_table is None:
                continue

            max_vif = vif_table["VIF"].max()
            mean_vif = vif_table["VIF"].mean()

            all_below_threshold = bool(
                (vif_table["VIF"] < threshold).all()
            )

            combination_id = len(combination_results) + 1

            combination_results.append({
                "combination_id": combination_id,
                "model": model_name,
                "n_variables": number_variables,
                "variables": " | ".join(combination),
                "max_vif": max_vif,
                "mean_vif": mean_vif,
                "all_vif_below_threshold": all_below_threshold
            })

            vif_details = vif_table.copy()
            vif_details.insert(
                0,
                "combination_id",
                combination_id
            )
            vif_details.insert(
                1,
                "model",
                model_name
            )
            detailed_vif_results.append(vif_details)

    results_df = pd.DataFrame(combination_results)

    if results_df.empty:
        raise ValueError(
            f"No valid combinations could be calculated for {model_name}."
        )

    detailed_df = pd.concat(
        detailed_vif_results,
        ignore_index=True
    )

    # Sort all combinations:
    # more variables first, then lower maximum and mean VIF
    results_df = results_df.sort_values(
        by=[
            "n_variables",
            "max_vif",
            "mean_vif"
        ],
        ascending=[
            False,
            True,
            True
        ]
    ).reset_index(drop=True)

    # Only combinations satisfying the threshold
    valid_results = results_df[
        results_df["all_vif_below_threshold"]
    ].copy()

    if not valid_results.empty:
        best_combination = valid_results.iloc[0]
        selection_note = (
            f"Best combination with all VIF values below {threshold}"
        )
    else:
        # Fallback: choose the overall combination with the lowest maximum VIF
        best_combination = (
            results_df
            .sort_values(
                by=["max_vif", "n_variables", "mean_vif"],
                ascending=[True, False, True]
            )
            .iloc[0]
        )

        selection_note = (
            f"No combination had all VIF values below {threshold}. "
            "Showing the combination with the lowest maximum VIF."
        )

    best_combination_id = int(
        best_combination["combination_id"]
    )

    best_vif_table = (
        detailed_df[
            detailed_df["combination_id"]
            == best_combination_id
        ]
        [["Variable", "VIF"]]
        .sort_values("VIF", ascending=False)
        .reset_index(drop=True)
    )

    print(f"\n{selection_note}")
    print("-" * 80)
    print(
        f"Number of selected variables: "
        f"{int(best_combination['n_variables'])}"
    )
    print(
        f"Maximum VIF: "
        f"{best_combination['max_vif']:.3f}"
    )
    print(
        f"Mean VIF: "
        f"{best_combination['mean_vif']:.3f}"
    )

    print("\nSelected variables:")
    for variable in best_combination["variables"].split(" | "):
        print(f"  - {variable}")

    print("\nVIF values:")
    print(
        best_vif_table.to_string(
            index=False,
            formatters={"VIF": "{:.3f}".format}
        )
    )

    return {
        "all_combinations": results_df,
        "all_vif_details": detailed_df,
        "valid_combinations": valid_results,
        "best_combination": best_combination,
        "best_vif_table": best_vif_table
    }


# =============================================================================
# RUN FOR BOTH MODELS
# =============================================================================

lst_results = test_all_vif_combinations(
    df=df_panel,
    variables=lst_variables,
    model_name="LST model",
    threshold=VIF_THRESHOLD,
    min_variables=MIN_VARIABLES
)

ndvi_results = test_all_vif_combinations(
    df=df_panel,
    variables=ndvi_variables,
    model_name="NDVI model",
    threshold=VIF_THRESHOLD,
    min_variables=MIN_VARIABLES
)

# =============================================================================
# SHOW TOP 5 VALID COMBINATIONS
# =============================================================================

def show_top5(results, model_name):
    """
    Display the five best valid VIF combinations for one model.
    """

    valid_results = results["valid_combinations"]
    detailed_df = results["all_vif_details"]

    print("\n" + "=" * 100)
    print(f"TOP 5 VALID COMBINATIONS — {model_name}")
    print("=" * 100)

    if valid_results.empty:
        print(
            f"No combination found with all VIF values below "
            f"{VIF_THRESHOLD}."
        )
        return

    top5 = (
        valid_results
        .sort_values(
            by=["n_variables", "max_vif", "mean_vif"],
            ascending=[False, True, True]
        )
        .head(5)
        .reset_index(drop=True)
    )

    # Short overview
    print(
        top5[
            [
                "combination_id",
                "n_variables",
                "max_vif",
                "mean_vif",
                "variables"
            ]
        ].to_string(
            index=False,
            formatters={
                "max_vif": "{:.3f}".format,
                "mean_vif": "{:.3f}".format
            }
        )
    )

    # Detailed VIF values for each combination
    for rank, row in top5.iterrows():
        combination_id = int(row["combination_id"])

        print("\n" + "-" * 100)
        print(
            f"Rank {rank + 1} | "
            f"Combination {combination_id} | "
            f"{int(row['n_variables'])} variables | "
            f"Maximum VIF: {row['max_vif']:.3f} | "
            f"Mean VIF: {row['mean_vif']:.3f}"
        )
        print("-" * 100)

        combination_vif = (
            detailed_df[
                detailed_df["combination_id"] == combination_id
            ][["Variable", "VIF"]]
            .sort_values("VIF", ascending=False)
            .reset_index(drop=True)
        )

        print(
            combination_vif.to_string(
                index=False,
                formatters={"VIF": "{:.3f}".format}
            )
        )


# =============================================================================
# DISPLAY RESULTS
# =============================================================================

show_top5(
    results=lst_results,
    model_name="LST model"
)

show_top5(
    results=ndvi_results,
    model_name="NDVI model"
)

# Linearity vs. Non-Linearity

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.stats import linregress
from statsmodels.nonparametric.smoothers_lowess import lowess


# =============================================================================
# CONFIGURATION
# =============================================================================

OUTPUT_DIR = Path(
    r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\Analysis_v2")


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LOWESS_FRAC = 0.40

# Only limits the displayed axis range.
# Observations are not removed from OLS or LOWESS estimation.
LIMIT_AXES_TO_PERCENTILES = False
LOWER_PERCENTILE = 0.01
UPPER_PERCENTILE = 0.99


# =============================================================================
# VARIABLES AND LABELS
# =============================================================================

lst_predictors = {
    "reldiff_avg_building_footprint":
        r"Rel. $\Delta$ Avg. Building Footprint [m$^2$]",
    "reldiff_avg_building_height":
        r"Rel. $\Delta$ Avg. Building Height [m]",
    "reldiff_built_up_ratio":
        r"Rel. $\Delta$ Built-up Ratio [%]",
    "diff_res_subclass_share_mfh_ab":
        r"$\Delta$ Share MFH-AB [%]",
    "difference_NDVI":
        r"$\Delta$NDVI"
}

ndvi_predictors = {
    "reldiff_avg_building_footprint":
        r"Rel. $\Delta$ Avg. Building Footprint [m$^2$]",
    "reldiff_avg_building_height":
        r"Rel. $\Delta$ Avg. Building Height [m]",
    "reldiff_built_up_ratio":
        r"Rel. $\Delta$ Built-up Ratio [%]",
    "diff_res_subclass_share_mfh_ab":
        r"$\Delta$ Share MFH-AB [%]"
}


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def prepare_data(df, x_variable, y_variable):
    """
    Prepare numeric complete cases for one predictor-response pair.
    """
    plot_df = (
        df[[x_variable, y_variable]]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    return plot_df


def add_ols_lowess_panel(
    ax,
    df,
    x_variable,
    y_variable,
    x_label,
    y_label,
    lowess_frac=0.40
):
    """
    Add scatter points, an OLS regression line and a LOWESS smoother
    to one matplotlib axis.
    """
    plot_df = prepare_data(
        df=df,
        x_variable=x_variable,
        y_variable=y_variable
    )

    if len(plot_df) < 10:
        ax.text(
            0.5,
            0.5,
            "Insufficient data",
            ha="center",
            va="center",
            transform=ax.transAxes
        )
        return

    x = plot_df[x_variable].to_numpy()
    y = plot_df[y_variable].to_numpy()

    # OLS line
    ols_result = linregress(x, y)

    x_grid = np.linspace(
        np.min(x),
        np.max(x),
        300
    )

    y_ols = (
        ols_result.intercept
        + ols_result.slope * x_grid
    )

    # LOWESS line
    lowess_result = lowess(
        endog=y,
        exog=x,
        frac=lowess_frac,
        it=3,
        return_sorted=True
    )

    # Scatter points
    ax.scatter(
        x,
        y,
        s=7,
        alpha=0.50,
        color="0.45",
        edgecolors="none",
        rasterized=True
    )

    # OLS regression line
    ax.plot(
        x_grid,
        y_ols,
        color="blue",
        linewidth=1.8
    )

    # LOWESS smoother
    ax.plot(
        lowess_result[:, 0],
        lowess_result[:, 1],
        color="red",
        linewidth=2.0
    )

    # Horizontal reference line at zero
    ax.axhline(
        0,
        color="0.5",
        linewidth=0.7,
        linestyle="-",
        alpha=0.45
    )

    # Labels and title
    ax.set_title(
        x_label,
        fontsize=12,
        fontweight="bold",
        loc="left",
        pad=6
    )

    ax.set_xlabel(
        x_label,
        fontsize=10
    )

    ax.set_ylabel(
        y_label,
        fontsize=10
    )

    # Light grid, similar to the example
    ax.grid(
        True,
        linewidth=0.6,
        alpha=0.25
    )

    ax.set_axisbelow(True)

    # Thin axis borders
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)
        spine.set_color("0.25")

    ax.tick_params(
        axis="both",
        labelsize=9,
        width=0.8
    )

    # Restrict only the displayed range
    if LIMIT_AXES_TO_PERCENTILES:
        x_lower = plot_df[x_variable].quantile(LOWER_PERCENTILE)
        x_upper = plot_df[x_variable].quantile(UPPER_PERCENTILE)

        y_lower = plot_df[y_variable].quantile(LOWER_PERCENTILE)
        y_upper = plot_df[y_variable].quantile(UPPER_PERCENTILE)

        x_padding = 0.05 * (x_upper - x_lower)
        y_padding = 0.05 * (y_upper - y_lower)

        ax.set_xlim(
            x_lower - x_padding,
            x_upper + x_padding
        )

        ax.set_ylim(
            y_lower - y_padding,
            y_upper + y_padding
        )


def create_panel_figure(
    df,
    predictors,
    response,
    response_label,
    panel_label,
    output_filename,
    ncols=2
):
    """
    Create a multi-panel figure containing one OLS/LOWESS plot
    for each explanatory variable.
    """
    n_plots = len(predictors)
    nrows = int(np.ceil(n_plots / ncols))

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(12, 4.2 * nrows),
        squeeze=False
    )

    axes = axes.flatten()

    for ax, (predictor, predictor_label) in zip(
        axes,
        predictors.items()
    ):
        add_ols_lowess_panel(
            ax=ax,
            df=df,
            x_variable=predictor,
            y_variable=response,
            x_label=predictor_label,
            y_label=response_label,
            lowess_frac=LOWESS_FRAC
        )

    # Remove unused axes
    for ax in axes[n_plots:]:
        fig.delaxes(ax)

    # Panel letter positioned like in the example
    fig.text(
        0.012,
        0.975,
        panel_label,
        fontsize=17,
        fontweight="bold",
        va="top",
        ha="left"
    )

    fig.tight_layout(
        rect=[0.035, 0.02, 1, 0.97],
        w_pad=2.5,
        h_pad=2.5
    )

    output_file = OUTPUT_DIR / output_filename

    fig.savefig(
        output_file,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    print(f"Saved figure to:\n{output_file}")


# =============================================================================
# CREATE LST FIGURE
# =============================================================================

create_panel_figure(
    df=df_panel,
    predictors=lst_predictors,
    response="difference_LST",
    response_label=r"$\Delta$LST",
    panel_label="a)",
    output_filename="LST_OLS_LOWESS_panels.png",
    ncols=2
)


# =============================================================================
# CREATE NDVI FIGURE
# =============================================================================

create_panel_figure(
    df=df_panel,
    predictors=ndvi_predictors,
    response="difference_NDVI",
    response_label=r"$\Delta$NDVI",
    panel_label="b)",
    output_filename="NDVI_OLS_LOWESS_panels.png",
    ncols=2
)